# 05 — Sales Forecasting
Facebook Prophet for time-series revenue forecasting.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sqlalchemy import create_engine
from src.forecast import load_daily_sales, run_prophet, compute_metrics
import warnings; warnings.filterwarnings('ignore')
import logging; logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

engine = create_engine('sqlite:///../data/ecommerce.db')
print("Ready")

## 1. Load & Visualise Raw Time Series

In [ ]:
df = load_daily_sales(engine)
print(df[['ds','y','orders','unique_customers']].tail(5).to_string(index=False))

fig, axes = plt.subplots(2,1,figsize=(14,8))
axes[0].fill_between(df['ds'], df['y']/1e3, alpha=0.4, color='#2980b9')
axes[0].plot(df['ds'], df['y']/1e3, color='#2980b9', lw=1.5)
axes[0].set_title('Daily Revenue (R$ thousands)'); axes[0].set_ylabel('Revenue (R$ 000s)')
axes[1].fill_between(df['ds'], df['orders'], alpha=0.4, color='#27ae60')
axes[1].plot(df['ds'], df['orders'], color='#27ae60', lw=1.5)
axes[1].set_title('Daily Order Count'); axes[1].set_ylabel('Orders')
for ax in axes: ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout(); plt.show()

## 2. Decompose Seasonality

In [ ]:
df2 = df.copy()
df2['dow']   = df2['ds'].dt.dayofweek
df2['month'] = df2['ds'].dt.month
dow_avg   = df2.groupby('dow')['y'].mean()
month_avg = df2.groupby('month')['y'].mean()

fig,axes = plt.subplots(1,2,figsize=(14,5))
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
axes[0].bar(day_names, dow_avg.values, color=['#e74c3c' if i>=5 else '#2980b9' for i in range(7)])
axes[0].set_title('Avg Revenue by Day of Week'); axes[0].set_ylabel('Avg Revenue (R$)')
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(month_names[:len(month_avg)], month_avg.values, color='#9b59b6')
axes[1].set_title('Avg Revenue by Month')
plt.tight_layout(); plt.show()

## 3. Fit Prophet & Forecast 90 Days

In [ ]:
model, forecast = run_prophet(df)
mape, rmse = compute_metrics(df, forecast)
print(f"Model MAPE: {mape:.2f}%")
print(f"Model RMSE: R$ {rmse:,.2f}")
forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(10).round(2)

## 4. Forecast Plot

In [ ]:
split = df['ds'].max()
hist = forecast[forecast['ds'] <= split]
fut  = forecast[forecast['ds'] >  split]

fig, ax = plt.subplots(figsize=(15,6))
ax.fill_between(forecast['ds'], forecast['yhat_lower']/1e3, forecast['yhat_upper']/1e3,
                alpha=0.15, color='#2980b9', label='80% confidence')
ax.plot(hist['ds'], hist['yhat']/1e3, color='#2980b9', lw=1.5, label='Prophet fit')
ax.plot(fut['ds'],  fut['yhat']/1e3,  color='#e74c3c', lw=2, linestyle='--', label='90-day forecast')
ax.scatter(df['ds'], df['y']/1e3, s=4, color='#333', alpha=0.4, label='Actual')
ax.axvline(split, color='gray', linestyle=':', lw=2, label='Forecast start')
ax.set_title('Revenue Forecast — Next 90 Days'); ax.set_ylabel('Revenue (R$ 000s)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.get_xticklabels(), rotation=30)
plt.tight_layout(); plt.show()

## 5. Components Plot

In [ ]:
fig = model.plot_components(forecast)
plt.suptitle('Prophet Decomposition: Trend + Seasonality', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

## 6. Export & Summary

In [ ]:
out = forecast[['ds','yhat','yhat_lower','yhat_upper']].merge(
    df[['ds','y','orders','unique_customers']], on='ds', how='left')
out.to_csv('../outputs/sales_forecast.csv', index=False)

future_fc = forecast[forecast['ds'] > split]
print(f"Next 30 days: R$ {future_fc.head(30)['yhat'].sum():,.0f}")
print(f"Next 60 days: R$ {future_fc.head(60)['yhat'].sum():,.0f}")
print(f"Next 90 days: R$ {future_fc.head(90)['yhat'].sum():,.0f}")
print(f"\nExported {len(out):,} rows → outputs/sales_forecast.csv")